# Module 2 — Feature Engineering
## Goal: Transform clean data into ML-ready format
## Steps:
1. Encode binary columns (Yes/No → 1/0)
2. Encode multi-category columns
3. Scale numerical features
4. Save ML-ready dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telco_churn_cleaned.csv')
print(f"Loaded: {df.shape}")
print(df.dtypes)

Loaded: (7032, 21)
customerID              str
gender                  str
SeniorCitizen           str
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


In [2]:
customer_ids = df['customerID'].copy()
df = df.drop(columns=['customerID'])

print("customerID saved separately")
print(f"Shape after drop: {df.shape}")

customerID saved separately
Shape after drop: (7032, 20)


In [3]:
# Churn: Yes → 1, No → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Churn encoding:")
print(df['Churn'].value_counts())

Churn encoding:
Churn
0    5163
1    1869
Name: count, dtype: int64


In [4]:
# All columns with Yes/No values
binary_cols = ['gender','Partner','Dependents','PhoneService',
               'MultipleLines','OnlineSecurity','OnlineBackup',
               'DeviceProtection','TechSupport','StreamingTV',
               'StreamingMovies','PaperlessBilling','SeniorCitizen']

# For columns with Yes/No/No internet service or similar
# We map Yes→1, everything else→0
for col in binary_cols:
    if df[col].nunique() == 2:
        # Simple binary
        vals = df[col].unique()
        df[col] = df[col].map({vals[0]: 0, vals[1]: 1})
    else:
        # Has third value like 'No internet service'
        df[col] = df[col].map({'Yes': 1}).fillna(0).astype(int)

print("Binary columns encoded:")
print(df[binary_cols].head())

Binary columns encoded:
   gender  Partner  Dependents  PhoneService  MultipleLines  OnlineSecurity  \
0       0        0           0             0              0               0   
1       1        1           0             1              0               1   
2       1        1           0             1              0               1   
3       1        1           0             0              0               1   
4       0        1           0             1              0               0   

   OnlineBackup  DeviceProtection  TechSupport  StreamingTV  StreamingMovies  \
0             1                 0            0            0                0   
1             0                 1            0            0                0   
2             1                 0            0            0                0   
3             0                 1            1            0                0   
4             0                 0            0            0                0   

   PaperlessBilling 

In [5]:
# InternetService: DSL, Fiber optic, No
# Contract: Month-to-month, One year, Two year
# PaymentMethod: 4 categories

multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=False)

print(f"Shape after encoding: {df.shape}")
print("New columns added:")
new_cols = [c for c in df.columns if any(m in c for m in multi_cat_cols)]
print(new_cols)

Shape after encoding: (7032, 27)
New columns added:
['InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [12]:
# Convert all boolean columns to integer (0/1)
bool_cols = df.select_dtypes(include='bool').columns.tolist()
print(f"Boolean columns found: {len(bool_cols)}")
print(bool_cols)

df[bool_cols] = df[bool_cols].astype(int)

print("\nAfter conversion — no boolean columns:")
print(df.select_dtypes(include='bool').columns.tolist())

Boolean columns found: 10
['InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

After conversion — no boolean columns:
[]


In [13]:
print("All columns in ML-ready dataset:")
for i, col in enumerate(df.columns):
    print(f"{i+1}. {col} — dtype: {df[col].dtype}")

All columns in ML-ready dataset:
1. customerID — dtype: str
2. gender — dtype: int64
3. SeniorCitizen — dtype: int64
4. Partner — dtype: int64
5. Dependents — dtype: int64
6. tenure — dtype: int64
7. PhoneService — dtype: int64
8. MultipleLines — dtype: int64
9. OnlineSecurity — dtype: int64
10. OnlineBackup — dtype: int64
11. DeviceProtection — dtype: int64
12. TechSupport — dtype: int64
13. StreamingTV — dtype: int64
14. StreamingMovies — dtype: int64
15. PaperlessBilling — dtype: int64
16. MonthlyCharges — dtype: float64
17. TotalCharges — dtype: float64
18. Churn — dtype: int64
19. InternetService_DSL — dtype: int64
20. InternetService_Fiber optic — dtype: int64
21. InternetService_No — dtype: int64
22. Contract_Month-to-month — dtype: int64
23. Contract_One year — dtype: int64
24. Contract_Two year — dtype: int64
25. PaymentMethod_Bank transfer (automatic) — dtype: int64
26. PaymentMethod_Credit card (automatic) — dtype: int64
27. PaymentMethod_Electronic check — dtype: int64
28. 

In [14]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")
print(f"\nChurn distribution:")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3) * 100)

Missing values: 0
Final shape: (7032, 28)

Churn distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64
Churn
0    73.4
1    26.6
Name: proportion, dtype: float64


In [11]:
# Save with customerID for tracking
df.insert(0, 'customerID', customer_ids.values)
df.to_csv('../data/telco_churn_ml_ready.csv', index=False)

print("ML-ready dataset saved to ../data/telco_churn_ml_ready.csv")
print(f"Final shape: {df.shape}")
df.head()

ML-ready dataset saved to ../data/telco_churn_ml_ready.csv
Final shape: (7032, 28)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,0,0,1,0,0,0,1,...,True,False,False,True,False,False,False,False,True,False
1,5575-GNVDE,1,0,1,0,34,1,0,1,0,...,True,False,False,False,True,False,False,False,False,True
2,3668-QPYBK,1,0,1,0,2,1,0,1,1,...,True,False,False,True,False,False,False,False,False,True
3,7795-CFOCW,1,0,1,0,45,0,0,1,0,...,True,False,False,False,True,False,True,False,False,False
4,9237-HQITU,0,0,1,0,2,1,0,0,0,...,False,True,False,True,False,False,False,False,True,False


## Feature Engineering Summary

| Step | Action | Result |
|---|---|---|
| Target encoding | Churn Yes/No → 1/0 | Binary target ready |
| Binary encoding | Yes/No columns → 1/0 | 13 columns encoded |
| One-hot encoding | InternetService, Contract, PaymentMethod | 3 columns → 10 columns |
| customerID | Preserved separately | For tracking later |

**Final dataset: 7,032 rows × ~32 columns — ready for model training**